In [ ]:
import pandas as pd
import os

folder_path = "public_data_dev/track_a/train"
dataframes = []


for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        df.columns = [col.lower() for col in df.columns]
        dataframes.append(df)


combined_df = pd.concat(dataframes, ignore_index=True)

if len(combined_df.columns) >= 6:
    for col in combined_df.columns[-6:]:
        combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce').fillna(0).astype(int)

df = combined_df.iloc[:,1:]

In [ ]:
df

In [ ]:
ID2LABEL = {}
LABEL2ID = {}

for idx,label in enumerate(df.columns):
    if label in ['text'] or label in ['id']:
        continue

    ID2LABEL[idx-1] = label
    LABEL2ID[label] = idx-1

print(f"ID2LABEL: {ID2LABEL}")
print(f"LABEL2ID: {LABEL2ID}")

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

# X = df['text']
# y = df.drop('text', axis=1)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# Combine them into a DatasetDict
dataset_dict = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

# Print information about the DatasetDict
print(dataset_dict)

In [ ]:
# get emotion counts by split type
split_types = list(dataset_dict.keys())
emotion_split_counts = {}

for label in LABEL2ID:
    for split_type in split_types:
        if label not in emotion_split_counts:
            emotion_split_counts[label] = []
        emotion_split_counts[label].append(sum(dataset_dict[split_type][label]))

print(f"SPLIT_TYPES: {split_types}")
print(f"EMOTION_SPLIT_COUNTS: {emotion_split_counts}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

LABEL2COLOR = {
    'anger': 'red',
    'disgust': 'green',
    'fear':'purple',
    'joy': 'yellow',
    'sadness': 'blue',
    'surprise': 'pink',
}

x = np.arange(len(split_types))
width = 0.15
multiplier = 0

fig, ax = plt.subplots()
for label, counts in emotion_split_counts.items():
    offset = width * multiplier
    rects = ax.bar(x + offset, counts, width, label=label, color=LABEL2COLOR[label])
    ax.bar_label(rects, label_type='edge')
    multiplier += 1

ax.set_xlabel('Split Type')
ax.set_ylabel('Count')
ax.set_title('Emotion Counts by Split Type')
ax.set_xticks(x + width, split_types)
ax.legend()
plt.show()

In [ ]:
emotion_counts = {}
for label in LABEL2ID:
    for split_type in dataset_dict.keys():
        emotion_counts[label] = emotion_counts.get(label,0)+sum(dataset_dict[split_type][label])

print(f"EMOTION_COUNTS: {emotion_counts}")

In [ ]:
# plot bar graph with total emotion counts
fig, ax = plt.subplots()
bar_container = ax.bar(emotion_counts.keys(), emotion_counts.values(), color=LABEL2COLOR.values())
ax.bar_label(bar_container, label_type='edge')
ax.set_xlabel('Emotion')
ax.set_ylabel('Count')
ax.set_title('Count by Emotions')
plt.show()

# Preprocessing

In [ ]:
def preprocess(batch):
    # rename column
    # batch['ID'] = batch['id']
    batch['Tweet'] = batch['text']

    # get one-hot encoded labels for each example in batch
    # for example: anger and sadness = vector of [1,0,0,0,1]
    batch['labels'] = [[float(batch[label][i]) for label in LABEL2ID] for i in range(len(batch['Tweet']))]
    return batch

preprocessed_datasets = dataset_dict.map(preprocess, batched=True, remove_columns=dataset_dict['train'].column_names)
preprocessed_datasets

In [ ]:
preprocessed_datasets['train'][2]

# Data Tokenization

In [ ]:
from transformers import AutoTokenizer

CHECKPOINT = 'bert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
tokenizer

In [ ]:
# tokenize out datasets with truncation
tokenized_datasets = preprocessed_datasets.map(lambda batch: tokenizer(batch['Tweet'], padding="max_length", truncation=True, max_length=512), batched=True, remove_columns=['Tweet'])
tokenized_datasets

In [ ]:
tokenized_datasets['train'][:1]

# Model Training

In [ ]:
# set seed for reproducibility
import torch

SEED = 42
torch.manual_seed(SEED)

In [ ]:
# let's clone a model and finetune as a multi-label classification problem
from transformers import AutoModelForSequenceClassification

# source: https://huggingface.co/distilbert-base-uncased
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, problem_type='multi_label_classification', num_labels=len(LABEL2ID), id2label=ID2LABEL, label2id=LABEL2ID)

In [16]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# this function calculates accuracy per label in a prediction instead of per prediction
def samples_accuracy_score(y_true, y_pred):
    return np.sum(y_true==y_pred) / y_true.size

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    # we sigmoid all logits for multilabel metrics
    predictions = torch.nn.functional.sigmoid(torch.Tensor(logits))
    # we set threshold to 0.50 to classify positive >= 0.50 and negative < 0.50
    predictions = (predictions >= 0.50).int().numpy()
    # overall accuracy measures accuracy of each true label list and prediction list
    overall_accuracy = accuracy_score(labels, predictions)
    # sample accuracy measures accuracy of each true label in a true label list and prediction in prediction list
    samples_accuracy = samples_accuracy_score(labels, predictions)
    # overall f1 measures macro f1 of each true label list and prediction list, ignoring zero division warnings
    overall_f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    # samples f1 measures f1 of each true label in a true label list and prediction in prediction list, ignoring zero division warnings
    samples_f1 = f1_score(labels, predictions, average='samples', zero_division=0)
    return {
        'overall_accuracy': overall_accuracy,
        'samples_accuracy': samples_accuracy,
        'overall_f1': overall_f1,
        'samples_f1': samples_f1,
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    seed=SEED,                          # seed for reproducibility
    output_dir='results',               # output directory to store epoch checkpoints
    num_train_epochs=5,                 # number of training epochs
    optim='adamw_torch',                # default optimizer as AdamW
    per_device_train_batch_size=32,     # 32 train batch size to speed up training
    per_device_eval_batch_size=32,      # 32 eval batch size to speed up evaluation
    evaluation_strategy='epoch',        # set evaluation strategy to each epoch instead of default 500 steps
    save_strategy='epoch',              # set saving of model strategy to each epoch instead of default 500 steps
    load_best_model_at_end=True,        # load the best model with lowest validation loss
    report_to='none',                   # suppress third-party logging
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
# let's see what an unfine-tuned bert can do
trainer.evaluate(tokenized_datasets['test'])

In [ ]:
# let's fine-tune bert as a multilabel problem
trainer.train()

In [ ]:
# let's see what a finetuned bert can do
trainer.evaluate(tokenized_datasets['test'])

In [22]:
predictions, label_ids, metrics = trainer.predict(tokenized_datasets['test'])

In [23]:
sigmoid = torch.sigmoid(torch.tensor(predictions))

In [24]:
predicted_labels = sigmoid.numpy()

In [25]:
threshold = 0.5
predicted_labels_binary = (predicted_labels > threshold).astype(int)

In [ ]:
for idx, (pred, true_label) in enumerate(zip(predicted_labels_binary, label_ids)):
    print(f"Row {idx}: Predicted labels = {pred}, True labels = {true_label}")

In [27]:
import pandas as pd
import os

folder_path = "public_data_dev/track_a/dev"
dataframes = []


for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        df.columns = [col.lower() for col in df.columns]
        dataframes.append(df)


combined_df = pd.concat(dataframes, ignore_index=True)

if len(combined_df.columns) >= 6:
    for col in combined_df.columns[-6:]:
        combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce').fillna(0).astype(int)

df = combined_df.iloc[:,1:]

In [32]:
trainer.save_model("model")

In [ ]:
from transformers import pipeline
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
import torch
import os
import pandas as pd

SEED = 42
torch.manual_seed(SEED)

CHECKPOINT = 'bert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

model = AutoModelForSequenceClassification.from_pretrained("model")
twitter_emotion_multilabel_classifier = pipeline(task='text-classification', model=model, tokenizer=tokenizer, device=torch.cuda.current_device(), top_k=None)

In [19]:
emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
def classify_emotions(tweet):
    # Run the pipeline on each tweet and get the labels
    results = twitter_emotion_multilabel_classifier(tweet)
    print(tweet)

    # Threshold for classification
    threshold = 0.5

    # Initialize binary labels with 0s for each emotion
    binary_labels = {emotion: 0 for emotion in emotion_labels}
    
    # Process the results from the pipeline
    for prediction in results[0]:
        label = prediction['label'].lower()  # Ensure lowercase to match the emotion labels
        score = prediction['score']

        # If the label score exceeds the threshold, set the corresponding emotion to 1
        if label in binary_labels and score > threshold:
            binary_labels[label] = 1
    print([binary_labels[emotion] for emotion in emotion_labels])
    # Return the binary labels as a list

    return [binary_labels[emotion] for emotion in emotion_labels]

In [ ]:
folder_path = "public_data_dev/track_a/dev"
dataframes = []

def tokenize_text(text):
    # Tokenize the text with truncation and padding to 512 tokens (if needed)
    encoding = tokenizer(text, truncation=True, padding='max_length', max_length=512, return_tensors="pt")
    return encoding

for file in os.listdir(folder_path):
    if file.endswith(".csv") and file=="swe.csv":
        file_path = os.path.join(folder_path, file)
        print(file)
        df = pd.read_csv(file_path)
        column_names = df.columns
        print("before",df.columns)
        df.drop(columns=df.columns[2:], inplace=True)

        df['text'] = df['text'].apply(lambda tweet: tokenize_text(tweet))

        df[emotion_labels] = pd.DataFrame(df['text'].apply(lambda tweet: classify_emotions(tweet)).to_list(), index=df.index)
        
        print("before",df.columns)

        # df.columns = [col.capitalize() if idx >= 2 else col for idx, col in enumerate(df.columns)]
        df = df[column_names]
        df.drop(columns="text", inplace=True)
        print("after",df.columns)
        df.to_csv(f'public_data_dev/track_a/pred/pred_{file}', index=False)

In [ ]:
folder_path = "public_data_dev/track_a/dev"
dataframes = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        print(file)
        df = pd.read_csv(file_path)
        column_names = df.columns
        print("before",df.columns)
        df.drop(columns=df.columns[2:], inplace=True)

        # df[emotion_labels] = pd.DataFrame(df['text'].apply(lambda tweet: classify_emotions(tweet)).to_list(), index=df.index)
        print(pd.DataFrame(df['text'].apply(lambda tweet: classify_emotions(tweet)).to_list(), index=df.index))
        print("before",df.columns)

        # df.columns = [col.capitalize() if idx >= 2 else col for idx, col in enumerate(df.columns)]
        df = df[column_names]
        df.drop(columns="text", inplace=True)
        print("after",df.columns)
        df.to_csv(f'public_data_dev/track_a/pred/pred_{file}', index=False)

In [ ]:
anger_tweet = """
We should lock the door and scream that curse word we know.
"""

twitter_emotion_multilabel_classifier(anger_tweet)

In [ ]:
disgust_tweet = """
You know what else barely touches the ground? Stray dogs, toenail clippings, road kill, hippies, dung beetles...
"""

twitter_emotion_multilabel_classifier(disgust_tweet)